# Taller SQL: Analítica Académica sobre `university.db`
**Especialización en Inteligencia Artificial** · SQLite + Pandas

Cada punto tiene: **código SQL**, **evidencia de ejecución** (tabla de resultados) e **interpretación analítica**.

**Notas técnicas**
1. `HORA_INICIO` y `HORA_FIN` son texto (`'08:00 AM'`); para detectar solapamientos se convierten a minutos desde la medianoche.
2. El último ID de usuario es 20, así que el siguiente libre es **21** (la guía menciona 22).
3. El DML (Bloque B) se ejecuta sobre una **copia** de la base para no alterar `university.db`.

## 0. Configuración

In [1]:
import sqlite3, shutil, os
import pandas as pd
pd.set_option('display.max_rows', 200, 'display.max_columns', 50, 'display.width', 200)

# En Colab: sube university.db cuando lo pida. En Antigravity/local: basta con tenerlo en la carpeta.
if not os.path.exists('university.db'):
    try:
        from google.colab import files
        files.upload()          # selecciona university.db
    except ImportError:
        raise FileNotFoundError('Coloca university.db en la misma carpeta del cuaderno.')

conn = sqlite3.connect('university.db')

def q(sql, conn=conn):
    """Ejecuta una consulta y devuelve un DataFrame."""
    return pd.read_sql_query(sql, conn)

q("SELECT name AS tabla FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")

,tabla
0,FACULTADES
1,EDIFICIOS
2,ROLES
3,CARRERAS
4,CURSOS
5,SALONES
6,USUARIOS
7,CALENDARIO_CURSOS
8,CARRERAS_ESTUDIANTES
9,CURSOS_CARRERAS


## Bloque A: Exploración, ordenamiento y filtros

### A1. Todos los registros de USUARIOS

In [2]:
sql = """
SELECT * FROM USUARIOS;
"""
q(sql)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,1,Ben,Stone,0,2
1,2,Grace,Stone,0,2
2,3,Mikaela,Stone,0,2
3,4,Jared,Vásquez,0,2
4,5,Zeke,Landon,0,2
5,6,Saanvi,Bahl,0,2
6,7,Ulrich,Nielsen,0,2
7,8,Charlotte,Doppler,0,2
8,9,Claudia,Tiedemann,0,2
9,10,Robert,Vance,0,2


**Interpretación:** La universidad tiene 20 usuarios: 10 profesores (semestre 0, rol 2) y 10 estudiantes (rol 1). El campo SEMESTRE vale 0 para docentes, por lo que cualquier promedio académico debe filtrar por rol.

### A2. NOMBRE y APELLIDO de los usuarios

In [3]:
sql = """
SELECT NOMBRE, APELLIDO FROM USUARIOS;
"""
q(sql)

,NOMBRE,APELLIDO
0,Ben,Stone
1,Grace,Stone
2,Mikaela,Stone
3,Jared,Vásquez
4,Zeke,Landon
5,Saanvi,Bahl
6,Ulrich,Nielsen
7,Charlotte,Doppler
8,Claudia,Tiedemann
9,Robert,Vance


**Interpretación:** Vista mínima de identificación. Útil para listados públicos de directorio sin exponer semestre ni rol.

### A3. Carreras en orden alfabético ascendente

In [4]:
sql = """
SELECT * FROM CARRERAS ORDER BY NOMBRE ASC;
"""
q(sql)

,ID_CARRERA,NOMBRE,ID_FACULTAD
0,4,Filosofía,2
1,2,Ingeniería Mecánica,1
2,1,Ingeniería de Sistemas,1
3,3,Matemáticas Aplicadas,1


**Interpretación:** Hay solo 4 programas: tres en Ingeniería y Ciencias y uno (Filosofía) en Humanidades. La oferta de pregrado es reducida y concentrada en una facultad.

### A4. Cursos en orden alfabético descendente

In [5]:
sql = """
SELECT * FROM CURSOS ORDER BY NOMBRE DESC;
"""
q(sql)

,ID_CURSO,NOMBRE,ID_FACULTAD
0,67,Ética,2.0
1,9,Árboles y Grafos,1.0
2,49,Álgebra Moderna,1.0
3,45,Álgebra Lineal,1.0
4,4,Técnicas y Prácticas de Programación,1.0
5,38,Transferencia de Calor,1.0
6,70,Trabajo de Grado,NaN
7,56,Textos Filosóficos,2.0
8,33,Termodinámica,1.0
9,50,Teoría de Probabilidades,1.0


**Interpretación:** El orden descendente en SQLite es por código de caracteres, así que las palabras con tilde (Árboles, Álgebra) quedan mal ubicadas, después de 'Ética'. Si se necesita orden lingüístico hay que normalizar acentos o usar una colación. También aparece 'Trabajo de Grado' con ID_FACULTAD NULL.

### A5. Estudiantes en semestre 2

In [6]:
sql = """
SELECT * FROM USUARIOS WHERE ID_ROL = 1 AND SEMESTRE = 2;
"""
q(sql)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,20,Mads,Nielsen,2,1


**Interpretación:** Solo un estudiante (Mads Nielsen, ID 20) cursa semestre 2: la cohorte de primer año es muy pequeña.

### A6. Usuarios con apellido 'Gomez'

In [7]:
sql = """
SELECT * FROM USUARIOS WHERE APELLIDO = 'Gomez';
"""
q(sql)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL


**Interpretación:** Ningún usuario tiene el apellido 'Gomez' (el ejemplo de la guía es genérico). Un resultado vacío es una respuesta válida: no es un error de sintaxis.

### A7. Estudiantes en semestre 1 o 2

In [8]:
sql = """
SELECT * FROM USUARIOS WHERE ID_ROL = 1 AND SEMESTRE IN (1, 2);
"""
q(sql)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,12,Cal,Stone,1,1
1,18,Elisabeth,Doppler,1,1
2,20,Mads,Nielsen,2,1


**Interpretación:** Tres estudiantes están en los semestres iniciales (Cal Stone, Elisabeth Doppler y Mads Nielsen). Dos de ellos no tienen matrículas (ver D1).

### A8. Asignaturas de Ingeniería y Ciencias relacionadas con inteligencia/datos

In [9]:
sql = """
SELECT ID_CURSO, NOMBRE
FROM CURSOS
WHERE ID_FACULTAD = 1
  AND (NOMBRE LIKE '%Inteligentes%' OR NOMBRE LIKE '%Aprendizaje%' OR NOMBRE LIKE '%Datos%')
ORDER BY NOMBRE;
"""
q(sql)

,ID_CURSO,NOMBRE
0,20,Aprendizaje Automático
1,14,Comunicación de Datos
2,3,Estructuras de Datos
3,17,Gestión y Modelación de Datos
4,24,Sistemas Inteligentes


**Interpretación:** Cinco asignaturas de la facultad 1 coinciden con los patrones. 'Estructuras de Datos', 'Aprendizaje Automático' y 'Sistemas Inteligentes' están programadas, pero las dos últimas no tienen estudiantes (ver D4); las otras dos no tienen horario (ver D2).

## Bloque B: Manipulación de datos (DML)

Se trabaja sobre una copia (`university_dml.db`).

In [10]:
shutil.copy('university.db', 'university_dml.db')
dml = sqlite3.connect('university_dml.db')

def run_dml(sql):
    dml.execute(sql); dml.commit()
    print('OK:', sql.strip().splitlines()[0], '...')

### B1. Insertar una nueva facultad

In [11]:
run_dml("""
INSERT INTO FACULTADES (ID_FACULTAD, NOMBRE)
VALUES (3, 'Artes y Diseño');
""")
q("""
SELECT * FROM FACULTADES;
""", dml)

OK: INSERT INTO FACULTADES (ID_FACULTAD, NOMBRE) ...


,ID_FACULTAD,NOMBRE
0,1,Ingeniería y Ciencias
1,2,Humanidades y Ciencias Sociales
2,3,Artes y Diseño


**Interpretación:** Se añadió la facultad con ID 3; ahora hay 3 filas.

### B2. Eliminar la facultad recién insertada

In [12]:
run_dml("""
DELETE FROM FACULTADES WHERE ID_FACULTAD = 3;
""")
q("SELECT * FROM FACULTADES" if "FACULTADES" in """DELETE FROM FACULTADES WHERE ID_FACULTAD = 3;""" else "SELECT * FROM USUARIOS WHERE ID_USUARIO = 21", dml)

OK: DELETE FROM FACULTADES WHERE ID_FACULTAD = 3; ...


,ID_FACULTAD,NOMBRE
0,1,Ingeniería y Ciencias
1,2,Humanidades y Ciencias Sociales


**Interpretación:** La facultad se eliminó; es seguro porque ninguna carrera ni curso la referencia.

### B3. Verificar que la facultad fue eliminada

In [13]:
q("""
SELECT * FROM FACULTADES WHERE ID_FACULTAD = 3;
""", dml)

,ID_FACULTAD,NOMBRE


**Interpretación:** Resultado vacío: el borrado se confirmó.

### B4. Insertar un nuevo estudiante (ID_ROL = 1)

In [14]:
run_dml("""
INSERT INTO USUARIOS (ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE, ID_ROL)
VALUES (21, 'Laura', 'Gomez', 1, 1);
""")
q("SELECT * FROM FACULTADES" if "FACULTADES" in """INSERT INTO USUARIOS (ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE, ID_ROL)
VALUES (21, 'Laura', 'Gomez', 1, 1);""" else "SELECT * FROM USUARIOS WHERE ID_USUARIO = 21", dml)

OK: INSERT INTO USUARIOS (ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE, ID_ROL) ...


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,21,Laura,Gomez,1,1


**Interpretación:** Registrada con ID 21 y rol de estudiante.

### B5. Verificar el nuevo registro

In [15]:
q("""
SELECT u.*, r.NOMBRE AS rol FROM USUARIOS u JOIN ROLES r ON r.ID_ROL = u.ID_ROL WHERE u.ID_USUARIO = 21;
""", dml)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL,rol
0,21,Laura,Gomez,1,1,Estudiante


**Interpretación:** Se guardó con el rol 'Estudiante'. Ahora la consulta A6 (apellido 'Gomez') ya devolvería una fila.

### B6. Actualizar el semestre del estudiante a 4

In [16]:
run_dml("""
UPDATE USUARIOS SET SEMESTRE = 4 WHERE ID_USUARIO = 21;
""")
q("""
SELECT * FROM USUARIOS WHERE ID_USUARIO = 21;
""", dml)

OK: UPDATE USUARIOS SET SEMESTRE = 4 WHERE ID_USUARIO = 21; ...


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,21,Laura,Gomez,4,1


**Interpretación:** El semestre cambió de 1 a 4. El WHERE con la clave primaria evita modificar otras filas.

### B7. Eliminar el estudiante ingresado

In [17]:
run_dml("""
DELETE FROM USUARIOS WHERE ID_USUARIO = 21;
""")
q("""
SELECT COUNT(*) AS total_usuarios FROM USUARIOS;
""", dml)

OK: DELETE FROM USUARIOS WHERE ID_USUARIO = 21; ...


,total_usuarios
0,20


**Interpretación:** El registro se eliminó y el conteo volvió a 20.

### B8. Verificar la eliminación del estudiante

In [18]:
q("""
SELECT * FROM USUARIOS WHERE ID_USUARIO = 21;
""", dml)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL


**Interpretación:** Resultado vacío: el estudiante ya no existe.

## Bloque C: Cruces relacionales (INNER JOIN)

### C1. Cada carrera con su facultad

In [19]:
sql = """
SELECT c.NOMBRE AS carrera, f.NOMBRE AS facultad
FROM CARRERAS c
INNER JOIN FACULTADES f ON f.ID_FACULTAD = c.ID_FACULTAD;
"""
q(sql)

,carrera,facultad
0,Ingeniería de Sistemas,Ingeniería y Ciencias
1,Ingeniería Mecánica,Ingeniería y Ciencias
2,Matemáticas Aplicadas,Ingeniería y Ciencias
3,Filosofía,Humanidades y Ciencias Sociales


**Interpretación:** Tres carreras dependen de Ingeniería y Ciencias y una de Humanidades. Es la base de cualquier reporte por facultad.

### C2. Usuarios con rol de estudiante

In [20]:
sql = """
SELECT u.NOMBRE, u.APELLIDO
FROM USUARIOS u
INNER JOIN ROLES r ON r.ID_ROL = u.ID_ROL
WHERE r.NOMBRE = 'Estudiante';
"""
q(sql)

,NOMBRE,APELLIDO
0,Olive,Stone
1,Cal,Stone
2,Angelina,Meyer
3,Jonas,Kahnwald
4,Martha,Nielsen
5,Bartosz,Tiedemann
6,Franziska,Doppler
7,Elisabeth,Doppler
8,Magnus,Nielsen
9,Mads,Nielsen


**Interpretación:** Diez estudiantes con rol 1. El JOIN con ROLES evita depender del número mágico del ID_ROL.

### C3. Usuarios con rol de profesor

In [21]:
sql = """
SELECT u.NOMBRE, u.APELLIDO
FROM USUARIOS u
INNER JOIN ROLES r ON r.ID_ROL = u.ID_ROL
WHERE r.NOMBRE = 'Profesor';
"""
q(sql)

,NOMBRE,APELLIDO
0,Ben,Stone
1,Grace,Stone
2,Mikaela,Stone
3,Jared,Vásquez
4,Zeke,Landon
5,Saanvi,Bahl
6,Ulrich,Nielsen
7,Charlotte,Doppler
8,Claudia,Tiedemann
9,Robert,Vance


**Interpretación:** Diez profesores con rol 2, mismo tamaño que la población estudiantil: relación 1 docente por estudiante.

### C4. Cursos matriculados por cada estudiante

In [22]:
sql = """
SELECT u.NOMBRE || ' ' || u.APELLIDO AS estudiante, cu.NOMBRE AS curso
FROM CURSOS_ESTUDIANTES ce
INNER JOIN USUARIOS u ON u.ID_USUARIO = ce.ID_USUARIO
INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
INNER JOIN CURSOS cu ON cu.ID_CURSO = cc.ID_CURSO
ORDER BY estudiante, curso;
"""
q(sql)

,estudiante,curso
0,Angelina Meyer,Filosofía Antigua
1,Angelina Meyer,Teología
2,Bartosz Tiedemann,Cinemática y Dinámica
3,Bartosz Tiedemann,Cinemática y Dinámica
4,Bartosz Tiedemann,Cinemática y Dinámica
5,Bartosz Tiedemann,Cálculo Diferencial
6,Bartosz Tiedemann,Cálculo Diferencial
7,Bartosz Tiedemann,Cálculo Integral
8,Bartosz Tiedemann,Cálculo Integral
9,Bartosz Tiedemann,Cálculo Multivariado


**Interpretación:** Hay 151 inscripciones. El mismo curso aparece varias veces por estudiante (p. ej. 'Cinemática y Dinámica' ×3 para Bartosz) porque cada fila es una sección distinta (Lunes, Miércoles, Viernes). Para listar cursos únicos hay que usar DISTINCT.

### C5. Cursos programados con salón y edificio

In [23]:
sql = """
SELECT cc.ID_CALENDARIO, cu.NOMBRE AS curso, s.NOMBRE AS salon, e.NOMBRE AS edificio
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS cu ON cu.ID_CURSO = cc.ID_CURSO
INNER JOIN SALONES s ON s.ID_SALON = cc.ID_SALON
INNER JOIN EDIFICIOS e ON e.ID_EDIFICIO = s.ID_EDIFICIO
ORDER BY cc.ID_CALENDARIO;
"""
q(sql)

,ID_CALENDARIO,curso,salon,edificio
0,1,Introducción a la Programación,1.4,El Lago
1,2,Introducción a la Programación,1.4,El Lago
2,3,Estructuras de Datos,3.4,Almendros
3,4,Estructuras de Datos,3.4,Almendros
4,5,Técnicas y Prácticas de Programación,3.4,Almendros
5,6,Técnicas y Prácticas de Programación,3.4,Almendros
6,7,Desarrollo de Videouegos,3.2,Palmas
7,8,Desarrollo de Videouegos,3.2,Palmas
8,9,Lógica Digital y Lenguaje de Máquina,1.2,El Lago
9,10,Lógica Digital y Lenguaje de Máquina,1.2,El Lago


**Interpretación:** 72 secciones programadas, con su salón y edificio. Sirve para validar que cada sección tenga espacio físico asignado: no hay NULLs.

### C6. Cartelera del día Martes

In [24]:
sql = """
SELECT cu.NOMBRE AS curso, cc.HORA_INICIO, cc.HORA_FIN,
       s.NOMBRE AS salon, e.NOMBRE AS edificio,
       p.NOMBRE || ' ' || p.APELLIDO AS profesor
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS cu    ON cu.ID_CURSO = cc.ID_CURSO
INNER JOIN SALONES s    ON s.ID_SALON = cc.ID_SALON
INNER JOIN EDIFICIOS e  ON e.ID_EDIFICIO = s.ID_EDIFICIO
INNER JOIN USUARIOS p   ON p.ID_USUARIO = cc.ID_PROFESOR
WHERE cc.DIA = 'Martes'
ORDER BY (CAST(substr(cc.HORA_INICIO,1,2) AS INTEGER) % 12 * 60 + CAST(substr(cc.HORA_INICIO,4,2) AS INTEGER)
       + CASE WHEN UPPER(cc.HORA_INICIO) LIKE '%PM' THEN 720 ELSE 0 END);
"""
q(sql)

,curso,HORA_INICIO,HORA_FIN,salon,edificio,profesor
0,Introducción a la Programación,08:00 AM,10:00 AM,1.4,El Lago,Ben Stone
1,Cálculo Diferencial,08:00 AM,10:00 AM,2.5,El Lago,Grace Stone
2,Antropología Filosófica,08:00 AM,10:00 AM,3.6,Almendros,Zeke Landon
3,Programación Paralela,08:00 AM,10:00 AM,3.3,Palmas,Ulrich Nielsen
4,Química y Ciencia de Materiales,08:00 AM,10:00 AM,3.6,El Lago,Claudia Tiedemann
5,Mecánica de Sólidos,09:00 AM,12:00 PM,2.3,El Lago,Saanvi Bahl
6,Estructuras de Datos,10:00 AM,12:00 PM,3.4,Almendros,Ben Stone
7,Álgebra Lineal,10:00 AM,12:00 PM,3.1,El Lago,Grace Stone
8,Cálculo Integral,10:00 AM,12:00 PM,3.3,El Lago,Jared Vásquez
9,Desarrollo de Videouegos,02:00 PM,05:00 PM,3.2,Palmas,Ben Stone


**Interpretación:** El martes hay 16 secciones. La franja de las 08:00 AM concentra la mayor carga, con 5 secciones simultáneas. Una de las horas tiene el formato inconsistente '04:00PM' (sin espacio), un problema de calidad de datos.

### C7. Horario completo del estudiante ID 17 (Franziska Doppler)

In [25]:
sql = """
SELECT cc.DIA, cc.HORA_INICIO, cc.HORA_FIN, cu.NOMBRE AS curso,
       s.NOMBRE AS salon, e.NOMBRE AS edificio
FROM CURSOS_ESTUDIANTES ce
INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
INNER JOIN CURSOS cu   ON cu.ID_CURSO = cc.ID_CURSO
INNER JOIN SALONES s   ON s.ID_SALON = cc.ID_SALON
INNER JOIN EDIFICIOS e ON e.ID_EDIFICIO = s.ID_EDIFICIO
WHERE ce.ID_USUARIO = 17
ORDER BY CASE cc.DIA WHEN 'Lunes' THEN 1 WHEN 'Martes' THEN 2 WHEN 'Miércoles' THEN 3
                     WHEN 'Jueves' THEN 4 WHEN 'Viernes' THEN 5 END,
         (CAST(substr(cc.HORA_INICIO,1,2) AS INTEGER) % 12 * 60 + CAST(substr(cc.HORA_INICIO,4,2) AS INTEGER)
       + CASE WHEN UPPER(cc.HORA_INICIO) LIKE '%PM' THEN 720 ELSE 0 END);
"""
q(sql)

,DIA,HORA_INICIO,HORA_FIN,curso,salon,edificio
0,Lunes,02:00 PM,04:00 PM,Cinemática y Dinámica,2.2,El Lago
1,Lunes,04:00 PM,06:00 PM,Técnicas y Prácticas de Programación,3.4,Almendros
2,Lunes,04:00 PM,06:00 PM,Arquitectura del Computador,2.6,El Lago
3,Martes,08:00 AM,10:00 AM,Introducción a la Programación,1.4,El Lago
4,Martes,08:00 AM,10:00 AM,Cálculo Diferencial,2.5,El Lago
5,Martes,10:00 AM,12:00 PM,Estructuras de Datos,3.4,Almendros
6,Martes,10:00 AM,12:00 PM,Álgebra Lineal,3.1,El Lago
7,Martes,10:00 AM,12:00 PM,Cálculo Integral,3.3,El Lago
8,Martes,02:00 PM,04:00 PM,Cálculo Multivariado,2.1,El Lago
9,Miércoles,02:00 PM,04:00 PM,Lógica Digital y Lenguaje de Máquina,1.2,El Lago


**Interpretación:** Franziska Doppler tiene 21 sesiones semanales, con varias en el mismo bloque horario (p. ej. Martes 08:00 AM: Introducción a la Programación y Cálculo Diferencial). Es un indicio de cruces que se confirma en el Reto 2.

### C8. Profesores y cantidad de secciones ofertadas (incluye los que no dictan nada)

In [26]:
sql = """
SELECT p.ID_USUARIO, p.NOMBRE || ' ' || p.APELLIDO AS profesor,
       COUNT(cc.ID_CALENDARIO) AS secciones_ofertadas
FROM USUARIOS p
LEFT JOIN CALENDARIO_CURSOS cc ON cc.ID_PROFESOR = p.ID_USUARIO
WHERE p.ID_ROL = 2
GROUP BY p.ID_USUARIO, profesor
ORDER BY secciones_ofertadas DESC;
"""
q(sql)

,ID_USUARIO,profesor,secciones_ofertadas
0,6,Saanvi Bahl,18
1,1,Ben Stone,8
2,2,Grace Stone,8
3,4,Jared Vásquez,8
4,9,Claudia Tiedemann,8
5,3,Mikaela Stone,6
6,10,Robert Vance,6
7,5,Zeke Landon,4
8,7,Ulrich Nielsen,4
9,8,Charlotte Doppler,2


**Interpretación:** Los 10 profesores tienen al menos una sección. Saanvi Bahl concentra 18 de las 72 (25 %), mientras Charlotte Doppler solo tiene 2: la carga docente está muy desbalanceada.

### C9. Profesores con los cursos ofertados que orientan

In [27]:
sql = """
SELECT DISTINCT p.NOMBRE || ' ' || p.APELLIDO AS profesor, cu.NOMBRE AS curso
FROM CALENDARIO_CURSOS cc
INNER JOIN USUARIOS p ON p.ID_USUARIO = cc.ID_PROFESOR
INNER JOIN CURSOS cu  ON cu.ID_CURSO = cc.ID_CURSO
ORDER BY profesor, curso;
"""
q(sql)

,profesor,curso
0,Ben Stone,Desarrollo de Videouegos
1,Ben Stone,Estructuras de Datos
2,Ben Stone,Introducción a la Programación
3,Ben Stone,Técnicas y Prácticas de Programación
4,Charlotte Doppler,Dibujo Mecánico
5,Claudia Tiedemann,Diseño Mecánico
6,Claudia Tiedemann,Máquinas Térmicas e Hidráulicas
7,Claudia Tiedemann,Propiedades de los Materiales
8,Claudia Tiedemann,Química y Ciencia de Materiales
9,Claudia Tiedemann,Álgebra Moderna


**Interpretación:** Una fila por par profesor-curso (40 combinaciones). Permite ver la especialización: Ben Stone, por ejemplo, orienta 4 cursos de programación.

## Bloque D: Auditoría de registros huérfanos

### D1. Estudiantes sin ninguna matrícula

In [28]:
sql = """
SELECT u.ID_USUARIO, u.NOMBRE, u.APELLIDO, u.SEMESTRE
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON ce.ID_USUARIO = u.ID_USUARIO
WHERE u.ID_ROL = 1 AND ce.ID_USUARIO IS NULL;
"""
q(sql)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
0,12,Cal,Stone,1
1,18,Elisabeth,Doppler,1


**Interpretación:** Cal Stone y Elisabeth Doppler son estudiantes de semestre 1 sin ninguna matrícula. Conviene contactarlos: pueden estar inactivos o tener la matrícula sin registrar.

### D2. Cursos del catálogo sin horario programado

In [29]:
sql = """
SELECT cu.ID_CURSO, cu.NOMBRE
FROM CURSOS cu
LEFT JOIN CALENDARIO_CURSOS cc ON cc.ID_CURSO = cu.ID_CURSO
WHERE cc.ID_CALENDARIO IS NULL
ORDER BY cu.ID_CURSO;
"""
q(sql)

,ID_CURSO,NOMBRE
0,2,Introducción al Modelado de Sistemas
1,5,Programación Orientada a Objetos
2,7,Programación Funcional
3,8,Lógica para Ciencias de la Computación
4,9,Árboles y Grafos
5,11,Diseño de Interfaces Humano-Computador
6,12,Computabilidad y Complejidad
7,13,Computación Gráfica
8,14,Comunicación de Datos
9,15,Desarrollo Formal de Sistemas


**Interpretación:** 35 de los 71 cursos del catálogo (49 %) no tienen ningún horario. Muchos son del núcleo de Sistemas (Programación Funcional, Árboles y Grafos, Sistemas Operativos): la malla existe en papel pero no en la programación.

### D3. Cursos del catálogo sin ningún estudiante matriculado

In [30]:
sql = """
SELECT cu.ID_CURSO, cu.NOMBRE
FROM CURSOS cu
WHERE cu.ID_CURSO NOT IN (
    SELECT cc.ID_CURSO
    FROM CALENDARIO_CURSOS cc
    INNER JOIN CURSOS_ESTUDIANTES ce ON ce.ID_CALENDARIO = cc.ID_CALENDARIO)
ORDER BY cu.ID_CURSO;
"""
q(sql)

,ID_CURSO,NOMBRE
0,2,Introducción al Modelado de Sistemas
1,5,Programación Orientada a Objetos
2,7,Programación Funcional
3,8,Lógica para Ciencias de la Computación
4,9,Árboles y Grafos
5,11,Diseño de Interfaces Humano-Computador
6,12,Computabilidad y Complejidad
7,13,Computación Gráfica
8,14,Comunicación de Datos
9,15,Desarrollo Formal de Sistemas


**Interpretación:** 48 de 71 cursos no tienen estudiantes matriculados. Solo 23 cursos (32 %) tienen demanda real.

### D4. Cursos ofertados (en calendario) sin estudiantes matriculados

In [31]:
sql = """
SELECT DISTINCT cu.ID_CURSO, cu.NOMBRE
FROM CURSOS cu
INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CURSO = cu.ID_CURSO
WHERE cu.ID_CURSO NOT IN (
    SELECT cc2.ID_CURSO
    FROM CALENDARIO_CURSOS cc2
    INNER JOIN CURSOS_ESTUDIANTES ce ON ce.ID_CALENDARIO = cc2.ID_CALENDARIO)
ORDER BY cu.ID_CURSO;
"""
q(sql)

,ID_CURSO,NOMBRE
0,19,Programación Paralela
1,20,Aprendizaje Automático
2,22,Introducción a la Seguridad Informática
3,24,Sistemas Inteligentes
4,26,Desarrollo de Videouegos
5,38,Transferencia de Calor
6,39,Diseño Mecánico
7,40,Máquinas Térmicas e Hidráulicas
8,52,Estadítica Aplicada
9,65,Antropología Filosófica


**Interpretación:** De los 48 sin matrículas, 35 ni siquiera se ofertan; los otros 13 sí están programados pero sin estudiantes (p. ej. Aprendizaje Automático y Sistemas Inteligentes). Es el punto de decisión: revisar si se abren, se fusionan o se promocionan.

## Bloque E: Agregaciones, estadísticas y cupos

### E1. Total de estudiantes

In [32]:
sql = """
SELECT COUNT(*) AS total_estudiantes FROM USUARIOS WHERE ID_ROL = 1;
"""
q(sql)

,total_estudiantes
0,10


**Interpretación:** La universidad registra 10 estudiantes.

### E2. Total de profesores y de estudiantes en una consulta

In [33]:
sql = """
SELECT SUM(CASE WHEN ID_ROL = 2 THEN 1 ELSE 0 END) AS total_profesores,
       SUM(CASE WHEN ID_ROL = 1 THEN 1 ELSE 0 END) AS total_estudiantes
FROM USUARIOS;
"""
q(sql)

,total_profesores,total_estudiantes
0,10,10


**Interpretación:** Se obtiene 10 profesores y 10 estudiantes en una sola pasada mediante agregación condicional (SUM + CASE), más eficiente que dos subconsultas.

### E3. Promedio de semestre de los estudiantes

In [34]:
sql = """
SELECT ROUND(AVG(SEMESTRE), 2) AS promedio_semestre FROM USUARIOS WHERE ID_ROL = 1;
"""
q(sql)

,promedio_semestre
0,4.0


**Interpretación:** El promedio de semestre es 4.0. Está calculado solo sobre estudiantes; si se incluyeran los docentes (semestre 0) bajaría a 2.0 y sería engañoso.

### E4. Estudiante(s) en el semestre más alto

In [35]:
sql = """
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE
FROM USUARIOS
WHERE ID_ROL = 1
  AND SEMESTRE = (SELECT MAX(SEMESTRE) FROM USUARIOS WHERE ID_ROL = 1);
"""
q(sql)

,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
0,19,Magnus,Nielsen,6


**Interpretación:** Magnus Nielsen (semestre 6) es el estudiante más avanzado y también el de mayor número de matrículas (29).

### E5. Cursos ofertados por facultad

In [36]:
sql = """
SELECT COALESCE(f.NOMBRE, 'Sin facultad asignada') AS facultad,
       COUNT(DISTINCT cc.ID_CURSO) AS cursos_ofertados,
       COUNT(cc.ID_CALENDARIO)     AS secciones
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS cu ON cu.ID_CURSO = cc.ID_CURSO
LEFT JOIN  FACULTADES f ON f.ID_FACULTAD = cu.ID_FACULTAD
GROUP BY facultad;
"""
q(sql)

,facultad,cursos_ofertados,secciones
0,Humanidades y Ciencias Sociales,5,6
1,Ingeniería y Ciencias,30,65
2,Sin facultad asignada,1,1


**Interpretación:** Ingeniería y Ciencias ofrece 30 cursos en 65 secciones; Humanidades 5 cursos en 6 secciones. Un curso ('Trabajo de Grado') no tiene facultad asignada: hay que corregir ese dato.

### E6. Facultades con más de 3 carreras

In [37]:
sql = """
SELECT f.NOMBRE, COUNT(*) AS num_carreras
FROM FACULTADES f
INNER JOIN CARRERAS c ON c.ID_FACULTAD = f.ID_FACULTAD
GROUP BY f.ID_FACULTAD, f.NOMBRE
HAVING COUNT(*) > 3;
"""
q(sql)

,NOMBRE,num_carreras


**Interpretación:** Ninguna facultad supera las 3 carreras (resultado vacío). La verificación E6b muestra que el máximo es 3 en Ingeniería y Ciencias: con HAVING > 2 sí aparecería.

### E6b. (Verificación) Carreras por facultad sin el filtro HAVING

In [38]:
sql = """
SELECT f.NOMBRE, COUNT(*) AS num_carreras
FROM FACULTADES f
INNER JOIN CARRERAS c ON c.ID_FACULTAD = f.ID_FACULTAD
GROUP BY f.ID_FACULTAD, f.NOMBRE;
"""
q(sql)

,NOMBRE,num_carreras
0,Ingeniería y Ciencias,3
1,Humanidades y Ciencias Sociales,1


**Interpretación:** Confirma el resultado vacío de E6: Ingeniería y Ciencias tiene 3 carreras y Humanidades 1, así que ninguna supera el umbral de 3.

### E7. Estudiantes matriculados por carrera

In [39]:
sql = """
SELECT c.NOMBRE AS carrera, COUNT(ce.ID_USUARIO) AS estudiantes
FROM CARRERAS c
LEFT JOIN CARRERAS_ESTUDIANTES ce ON ce.ID_CARRERA = c.ID_CARRERA
GROUP BY c.ID_CARRERA, c.NOMBRE
ORDER BY estudiantes DESC;
"""
q(sql)

,carrera,estudiantes
0,Ingeniería Mecánica,4
1,Ingeniería de Sistemas,3
2,Matemáticas Aplicadas,2
3,Filosofía,1


**Interpretación:** Ingeniería Mecánica lidera con 4 estudiantes, seguida de Sistemas con 3, Matemáticas con 2 y Filosofía con 1.

### E8. Carrera con más estudiantes

In [40]:
sql = """
SELECT c.NOMBRE AS carrera, COUNT(*) AS estudiantes
FROM CARRERAS c
INNER JOIN CARRERAS_ESTUDIANTES ce ON ce.ID_CARRERA = c.ID_CARRERA
GROUP BY c.ID_CARRERA, c.NOMBRE
HAVING COUNT(*) = (SELECT MAX(n) FROM (SELECT COUNT(*) AS n FROM CARRERAS_ESTUDIANTES GROUP BY ID_CARRERA));
"""
q(sql)

,carrera,estudiantes
0,Ingeniería Mecánica,4


**Interpretación:** Ingeniería Mecánica es la carrera con más estudiantes (4). Se usa un HAVING contra el máximo para que, si hubiera empate, devuelva todas las carreras empatadas.

### E9. Edificio con más salones

In [41]:
sql = """
SELECT e.NOMBRE AS edificio, COUNT(*) AS salones
FROM EDIFICIOS e
INNER JOIN SALONES s ON s.ID_EDIFICIO = e.ID_EDIFICIO
GROUP BY e.ID_EDIFICIO, e.NOMBRE
HAVING COUNT(*) = (SELECT MAX(n) FROM (SELECT COUNT(*) AS n FROM SALONES GROUP BY ID_EDIFICIO));
"""
q(sql)

,edificio,salones
0,El Lago,18


**Interpretación:** El Lago tiene 18 de los 42 salones (43 %), así que concentra la infraestructura; es el edificio a priorizar en mantenimiento y control de ocupación.

### E10. Cursos ofertados (secciones) por docente, de mayor a menor

In [42]:
sql = """
SELECT p.NOMBRE || ' ' || p.APELLIDO AS profesor,
       COUNT(*) AS secciones,
       COUNT(DISTINCT cc.ID_CURSO) AS cursos_distintos
FROM CALENDARIO_CURSOS cc
INNER JOIN USUARIOS p ON p.ID_USUARIO = cc.ID_PROFESOR
GROUP BY p.ID_USUARIO, profesor
ORDER BY secciones DESC, profesor;
"""
q(sql)

,profesor,secciones,cursos_distintos
0,Saanvi Bahl,18,8
1,Ben Stone,8,4
2,Claudia Tiedemann,8,5
3,Grace Stone,8,4
4,Jared Vásquez,8,5
5,Mikaela Stone,6,4
6,Robert Vance,6,3
7,Ulrich Nielsen,4,2
8,Zeke Landon,4,4
9,Charlotte Doppler,2,1


**Interpretación:** Saanvi Bahl dicta 18 secciones de 8 cursos distintos (25 % de la oferta). Es un riesgo operativo: una ausencia afectaría un cuarto de la programación.

### E11. Número de matrículas por estudiante

In [43]:
sql = """
SELECT u.ID_USUARIO, u.NOMBRE || ' ' || u.APELLIDO AS estudiante, COUNT(ce.ID_CALENDARIO) AS matriculas
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON ce.ID_USUARIO = u.ID_USUARIO
WHERE u.ID_ROL = 1
GROUP BY u.ID_USUARIO, estudiante
ORDER BY matriculas DESC;
"""
q(sql)

,ID_USUARIO,estudiante,matriculas
0,19,Magnus Nielsen,29
1,14,Jonas Kahnwald,28
2,16,Bartosz Tiedemann,28
3,17,Franziska Doppler,21
4,11,Olive Stone,19
5,15,Martha Nielsen,19
6,20,Mads Nielsen,5
7,13,Angelina Meyer,2
8,12,Cal Stone,0
9,18,Elisabeth Doppler,0


**Interpretación:** Magnus, Jonas y Bartosz tienen 28-29 matrículas, y Franziska 21. Dos estudiantes no tienen ninguna.

### E12. Estudiantes matriculados en menos de 10 cursos

In [44]:
sql = """
SELECT u.ID_USUARIO, u.NOMBRE || ' ' || u.APELLIDO AS estudiante, COUNT(ce.ID_CALENDARIO) AS matriculas
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON ce.ID_USUARIO = u.ID_USUARIO
WHERE u.ID_ROL = 1
GROUP BY u.ID_USUARIO, estudiante
HAVING COUNT(ce.ID_CALENDARIO) < 10
ORDER BY matriculas;
"""
q(sql)

,ID_USUARIO,estudiante,matriculas
0,12,Cal Stone,0
1,18,Elisabeth Doppler,0
2,13,Angelina Meyer,2
3,20,Mads Nielsen,5


**Interpretación:** Cuatro estudiantes están por debajo de 10 matrículas: 2 con 0, uno con 2 (Angelina Meyer) y uno con 5 (Mads Nielsen). Son candidatos a seguimiento académico por baja carga.

### E13. Curso con más matrículas vs. curso con más estudiantes únicos

In [45]:
sql = """
WITH por_curso AS (
    SELECT cu.ID_CURSO, cu.NOMBRE,
           COUNT(*)                     AS matriculas,
           COUNT(DISTINCT ce.ID_USUARIO) AS estudiantes_unicos
    FROM CURSOS_ESTUDIANTES ce
    INNER JOIN CALENDARIO_CURSOS cc ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
    INNER JOIN CURSOS cu ON cu.ID_CURSO = cc.ID_CURSO
    GROUP BY cu.ID_CURSO, cu.NOMBRE
)
SELECT 'Más matrículas' AS criterio, NOMBRE, matriculas, estudiantes_unicos
FROM por_curso WHERE matriculas = (SELECT MAX(matriculas) FROM por_curso)
UNION ALL
SELECT 'Más estudiantes únicos', NOMBRE, matriculas, estudiantes_unicos
FROM por_curso WHERE estudiantes_unicos = (SELECT MAX(estudiantes_unicos) FROM por_curso);
"""
q(sql)

,criterio,NOMBRE,matriculas,estudiantes_unicos
0,Más matrículas,Cinemática y Dinámica,18,6
1,Más estudiantes únicos,Cálculo Diferencial,14,7


**Interpretación:** 'Cinemática y Dinámica' lidera en matrículas (18) porque se dicta 3 veces por semana, pero 'Cálculo Diferencial' tiene más estudiantes distintos (7). Medir por matrículas sesga hacia cursos con más secciones; para demanda real conviene usar estudiantes únicos.

### E14. Demanda y cupos: secciones con 5 o más inscritos

In [46]:
sql = """
SELECT cc.ID_CALENDARIO, cu.NOMBRE AS curso, cc.DIA, cc.CUPO,
       COUNT(ce.ID_USUARIO) AS inscritos
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS cu ON cu.ID_CURSO = cc.ID_CURSO
INNER JOIN CURSOS_ESTUDIANTES ce ON ce.ID_CALENDARIO = cc.ID_CALENDARIO
GROUP BY cc.ID_CALENDARIO, cu.NOMBRE, cc.DIA, cc.CUPO
HAVING COUNT(ce.ID_USUARIO) >= 5
ORDER BY inscritos DESC;
"""
q(sql)

,ID_CALENDARIO,curso,DIA,CUPO,inscritos
0,13,Cálculo Diferencial,Martes,30,7
1,14,Cálculo Diferencial,Jueves,30,7
2,15,Álgebra Lineal,Martes,30,6
3,16,Álgebra Lineal,Jueves,30,6
4,23,Cálculo Integral,Martes,20,6
5,24,Cálculo Integral,Jueves,20,6
6,25,Cálculo Multivariado,Martes,20,6
7,26,Cálculo Multivariado,Jueves,20,6
8,31,Teología,Lunes,30,6
9,35,Cinemática y Dinámica,Lunes,30,6


**Interpretación:** 15 secciones tienen 5 o más inscritos, y ninguna se acerca al cupo (20-30). La ocupación máxima es 7/30 (23 %): hay capacidad sobrante, lo que permitiría consolidar secciones.

## Parte II: Retos integradores

### R1. Reto 1: carga académica por carrera

In [47]:
sql = """
SELECT c.NOMBRE AS carrera,
       f.NOMBRE AS facultad,
       (SELECT COUNT(*) FROM CURSOS_CARRERAS cc WHERE cc.ID_CARRERA = c.ID_CARRERA)       AS cursos_en_malla,
       (SELECT COUNT(*) FROM CARRERAS_ESTUDIANTES ce WHERE ce.ID_CARRERA = c.ID_CARRERA)  AS estudiantes
FROM CARRERAS c
INNER JOIN FACULTADES f ON f.ID_FACULTAD = c.ID_FACULTAD
ORDER BY c.NOMBRE;
"""
q(sql)

,carrera,facultad,cursos_en_malla,estudiantes
0,Filosofía,Humanidades y Ciencias Sociales,17,1
1,Ingeniería Mecánica,Ingeniería y Ciencias,25,4
2,Ingeniería de Sistemas,Ingeniería y Ciencias,37,3
3,Matemáticas Aplicadas,Ingeniería y Ciencias,22,2


**Interpretación:** Sistemas tiene la malla más larga (37 cursos), pero solo 3 estudiantes: la oferta curricular es más grande que la demanda. Mecánica tiene 25 cursos y 4 estudiantes, y Filosofía 17 cursos y 1. Se usaron subconsultas escalares para evitar el producto cartesiano.

### R1b. (Demostración) Mismo reto con JOIN doble sin DISTINCT: conteos inflados

In [48]:
sql = """
SELECT c.NOMBRE AS carrera,
       COUNT(cc.ID_CURSO)  AS cursos_inflado,
       COUNT(ce.ID_USUARIO) AS estudiantes_inflado
FROM CARRERAS c
LEFT JOIN CURSOS_CARRERAS cc ON cc.ID_CARRERA = c.ID_CARRERA
LEFT JOIN CARRERAS_ESTUDIANTES ce ON ce.ID_CARRERA = c.ID_CARRERA
GROUP BY c.ID_CARRERA, c.NOMBRE;
"""
q(sql)

,carrera,cursos_inflado,estudiantes_inflado
0,Ingeniería de Sistemas,111,111
1,Ingeniería Mecánica,100,100
2,Matemáticas Aplicadas,44,44
3,Filosofía,17,17


**Interpretación:** Demostración del error. Un JOIN doble multiplica filas: Sistemas pasa de 37 cursos y 3 estudiantes a 111 y 111 (37 × 3). Los resultados son idénticos entre sí, una señal de alerta.

### Reto 2: Auditoría Text-to-SQL (cruces de horario)

> **Cómo usar esta sección:** pídele al agente de Antigravity *"Identifica qué estudiantes tienen cruces de horario (dos clases el mismo día y a la misma hora) en university_db"*. **Pega su consulta literal en la celda a)** (reemplazando el ejemplo) y compárala con la versión validada de la celda b). La consulta de la celda a) es un **ejemplo típico** de lo que genera un agente, no la salida real del tuyo.

#### a) Consulta del agente (ejemplo ingenuo, reemplázala por la tuya)

In [49]:
sql_agente = """
SELECT DISTINCT u.NOMBRE, u.APELLIDO
FROM USUARIOS u
JOIN CURSOS_ESTUDIANTES ce1 ON ce1.ID_USUARIO = u.ID_USUARIO
JOIN CURSOS_ESTUDIANTES ce2 ON ce2.ID_USUARIO = u.ID_USUARIO
JOIN CALENDARIO_CURSOS c1 ON c1.ID_CALENDARIO = ce1.ID_CALENDARIO
JOIN CALENDARIO_CURSOS c2 ON c2.ID_CALENDARIO = ce2.ID_CALENDARIO
WHERE c1.DIA = c2.DIA
  AND c1.HORA_INICIO = c2.HORA_INICIO;
"""
q(sql_agente)

,NOMBRE,APELLIDO
0,Franziska,Doppler
1,Mads,Nielsen
2,Olive,Stone
3,Jonas,Kahnwald
4,Martha,Nielsen
5,Bartosz,Tiedemann
6,Magnus,Nielsen
7,Angelina,Meyer


Devuelve 8 estudiantes, incluida **Angelina Meyer**, que solo tiene 2 matrículas sin cruce real (falso positivo).

#### b) Consulta validada
Compara cada par de secciones del mismo estudiante (`x.ID_CALENDARIO < y.ID_CALENDARIO`) y aplica `inicio_A < fin_B AND inicio_B < fin_A` con las horas convertidas a minutos.

In [50]:
sql_correcta = """
SELECT u.ID_USUARIO, u.NOMBRE || ' ' || u.APELLIDO AS estudiante,
       a.ID_CALENDARIO AS seccion_a, ca.NOMBRE AS curso_a, a.DIA, a.HORA_INICIO AS ini_a, a.HORA_FIN AS fin_a,
       b.ID_CALENDARIO AS seccion_b, cb.NOMBRE AS curso_b, b.HORA_INICIO AS ini_b, b.HORA_FIN AS fin_b
FROM CURSOS_ESTUDIANTES x
INNER JOIN CURSOS_ESTUDIANTES y ON y.ID_USUARIO = x.ID_USUARIO
                               AND x.ID_CALENDARIO < y.ID_CALENDARIO
INNER JOIN CALENDARIO_CURSOS a ON a.ID_CALENDARIO = x.ID_CALENDARIO
INNER JOIN CALENDARIO_CURSOS b ON b.ID_CALENDARIO = y.ID_CALENDARIO
INNER JOIN CURSOS ca ON ca.ID_CURSO = a.ID_CURSO
INNER JOIN CURSOS cb ON cb.ID_CURSO = b.ID_CURSO
INNER JOIN USUARIOS u ON u.ID_USUARIO = x.ID_USUARIO
WHERE a.DIA = b.DIA
  AND (CAST(substr(a.HORA_INICIO,1,2) AS INTEGER) % 12 * 60 + CAST(substr(a.HORA_INICIO,4,2) AS INTEGER)
       + CASE WHEN UPPER(a.HORA_INICIO) LIKE '%PM' THEN 720 ELSE 0 END) < (CAST(substr(b.HORA_FIN,1,2) AS INTEGER) % 12 * 60 + CAST(substr(b.HORA_FIN,4,2) AS INTEGER)
       + CASE WHEN UPPER(b.HORA_FIN) LIKE '%PM' THEN 720 ELSE 0 END)
  AND (CAST(substr(b.HORA_INICIO,1,2) AS INTEGER) % 12 * 60 + CAST(substr(b.HORA_INICIO,4,2) AS INTEGER)
       + CASE WHEN UPPER(b.HORA_INICIO) LIKE '%PM' THEN 720 ELSE 0 END) < (CAST(substr(a.HORA_FIN,1,2) AS INTEGER) % 12 * 60 + CAST(substr(a.HORA_FIN,4,2) AS INTEGER)
       + CASE WHEN UPPER(a.HORA_FIN) LIKE '%PM' THEN 720 ELSE 0 END)
ORDER BY u.ID_USUARIO, a.DIA;
"""
df_cruces = q(sql_correcta)
print("Pares en conflicto:", len(df_cruces))
df_cruces

Pares en conflicto: 70


,ID_USUARIO,estudiante,seccion_a,curso_a,DIA,ini_a,fin_a,seccion_b,curso_b,ini_b,fin_b
0,11,Olive Stone,16,Álgebra Lineal,Jueves,10:00 AM,12:00 PM,24,Cálculo Integral,10:00 AM,12:00 PM
1,11,Olive Stone,19,Teoría de Probabilidades,Lunes,08:00 AM,10:00 AM,38,Física Térmica y Ondulatoria,08:00 AM,10:00 AM
2,11,Olive Stone,15,Álgebra Lineal,Martes,10:00 AM,12:00 PM,23,Cálculo Integral,10:00 AM,12:00 PM
3,11,Olive Stone,20,Teoría de Probabilidades,Miércoles,08:00 AM,10:00 AM,39,Física Térmica y Ondulatoria,08:00 AM,10:00 AM
4,14,Jonas Kahnwald,16,Álgebra Lineal,Jueves,10:00 AM,12:00 PM,24,Cálculo Integral,10:00 AM,12:00 PM
5,14,Jonas Kahnwald,16,Álgebra Lineal,Jueves,10:00 AM,12:00 PM,58,Dibujo Mecánico,10:00 AM,12:00 PM
6,14,Jonas Kahnwald,24,Cálculo Integral,Jueves,10:00 AM,12:00 PM,58,Dibujo Mecánico,10:00 AM,12:00 PM
7,14,Jonas Kahnwald,14,Cálculo Diferencial,Jueves,08:00 AM,10:00 AM,60,Química y Ciencia de Materiales,08:00 AM,10:00 AM
8,14,Jonas Kahnwald,26,Cálculo Multivariado,Jueves,02:00 PM,04:00 PM,62,Propiedades de los Materiales,02:00 PM,04:00 PM
9,14,Jonas Kahnwald,31,Teología,Lunes,10:00 AM,12:00 PM,41,Termodinámica,10:00 AM,12:00 PM


#### c) Resumen por estudiante

In [51]:
df_cruces.groupby('estudiante').size().sort_values(ascending=False).rename('pares_en_conflicto').to_frame()

,pares_en_conflicto
estudiante,
Magnus Nielsen,20
Jonas Kahnwald,15
Bartosz Tiedemann,15
Franziska Doppler,11
Martha Nielsen,4
Olive Stone,4
Mads Nielsen,1


#### d) Comparación cuantitativa de las dos versiones

In [52]:
sql_fix = """
SELECT DISTINCT u.NOMBRE, u.APELLIDO
FROM USUARIOS u
JOIN CURSOS_ESTUDIANTES ce1 ON ce1.ID_USUARIO = u.ID_USUARIO
JOIN CURSOS_ESTUDIANTES ce2 ON ce2.ID_USUARIO = u.ID_USUARIO
JOIN CALENDARIO_CURSOS c1 ON c1.ID_CALENDARIO = ce1.ID_CALENDARIO
JOIN CALENDARIO_CURSOS c2 ON c2.ID_CALENDARIO = ce2.ID_CALENDARIO
WHERE ce1.ID_CALENDARIO <> ce2.ID_CALENDARIO
  AND c1.DIA = c2.DIA
  AND c1.HORA_INICIO = c2.HORA_INICIO;
"""
pares_ingenuo = q("""
SELECT COUNT(*) AS n FROM CURSOS_ESTUDIANTES x
JOIN CURSOS_ESTUDIANTES y ON y.ID_USUARIO=x.ID_USUARIO AND x.ID_CALENDARIO<y.ID_CALENDARIO
JOIN CALENDARIO_CURSOS a ON a.ID_CALENDARIO=x.ID_CALENDARIO
JOIN CALENDARIO_CURSOS b ON b.ID_CALENDARIO=y.ID_CALENDARIO
WHERE a.DIA=b.DIA AND a.HORA_INICIO=b.HORA_INICIO""").iloc[0,0]
print("Estudiantes - agente ingenuo:", len(q(sql_agente)))
print("Estudiantes - agente con <> arreglado:", len(q(sql_fix)))
print("Estudiantes - consulta validada:", df_cruces['ID_USUARIO'].nunique())
print(f"Pares detectados por igualdad de hora de inicio: {pares_ingenuo} de {len(df_cruces)}")

Estudiantes - agente ingenuo: 8
Estudiantes - agente con <> arreglado: 7
Estudiantes - consulta validada: 7
Pares detectados por igualdad de hora de inicio: 53 de 70


#### e) Análisis crítico

| # | Posible error | ¿Ocurre en el ejemplo? | Efecto |
|---|---|---|---|
| 1 | **Comparar una sección consigo misma** (falta `ce1.ID_CALENDARIO <> ce2.ID_CALENDARIO`) | **Sí** | Todo estudiante con al menos una clase se marca como conflicto; aparece Angelina Meyer como falso positivo. |
| 2 | **Ignorar solapamientos parciales** (compara `HORA_INICIO` por igualdad) | **Sí** | No detecta, por ejemplo, *Mecánica de Sólidos* (09:00-12:00) frente a *Álgebra Lineal* (10:00-12:00). |
| 3 | **Comparar horas como texto** | Latente | Con `<` o `>` falla (`'02:00 PM' < '10:00 AM'`); además hay un `'04:00PM'` sin espacio. |
| 4 | **Duplicados simétricos** (A,B) y (B,A) | Ocultos por `DISTINCT` | Si se cuentan pares, se duplican; se evita con `<`. |

Corrigiendo solo la autocomparación, el ejemplo devuelve los 7 estudiantes correctos pero detecta únicamente 53 de los 70 pares: 17 solapamientos parciales quedan sin detectar. Es el caso más peligroso del Text-to-SQL: no lanza error y el resultado parece razonable.

**Interpretación para gestión académica:** 7 de los 8 estudiantes con matrículas tienen cruces. Magnus Nielsen (20 pares), Jonas Kahnwald y Bartosz Tiedemann (15 cada uno) concentran la mayoría. Los patrones se repiten (por ejemplo, Álgebra Lineal con Cálculo Integral los martes y jueves a las 10:00 AM), lo que apunta a un problema de **programación de la oferta** más que de decisiones individuales. Se recomienda mover una de las dos secciones o bloquear la matrícula conjunta.

**Lecciones:** (1) pedir al agente que explique la lógica del solapamiento; (2) validar contra un caso conocido; (3) revisar tipos de datos antes de comparar; (4) contrastar conteos entre dos versiones de la consulta.

## Resumen ejecutivo

- 10 profesores, 10 estudiantes, 4 carreras, 71 cursos y 72 secciones.
- 48 de 71 cursos no tienen matrículas; 35 ni siquiera tienen horario.
- Ocupación máxima por sección: 7 inscritos sobre un cupo de 30.
- Saanvi Bahl dicta 18 de 72 secciones (25 %).
- 70 pares de secciones en conflicto en 7 estudiantes.
- Calidad de datos: un curso sin facultad (`Trabajo de Grado`), una hora con formato distinto (`04:00PM`) y horas guardadas como texto.